<a href="https://colab.research.google.com/github/FadouaBOUAFIF/Cours_Computer_Vision_TP/blob/main/TP3_Filtrage_Spatial_Contours_SIAA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## Objectifs
À l'issue de cette séance de travaux pratiques, l'étudiant sera capable de :
1. **Implémenter les opérateurs de **corrélation croisée** ($g = h \otimes f$) et de **convolution 2D** ($g = h * f$) avec gestion des conditions aux bords (`zero-padding`, `reflect`, `replicate`).
2. **Modéliser et discrétiser analytiquement** un noyau Gaussien 2D $G(x,y) = \frac{1}{2\pi\sigma^2}e^{-\frac{x^2+y^2}{2\sigma^2}}$ et vérifier expérimentalement la **propriété de séparabilité** ($G(x,y) = G(x) \cdot G(y)$) en termes de précision numérique et de complexité computationnelle.
3. **Implémenter et comparer** les filtres de **rehaussement haute fréquence** (Laplacien isotrope $L_{-4}, L_{-8}$ et *Unsharp Masking*) ainsi que les **opérateurs de Sobel** (magnitude du gradient et carte d'orientation).
4. **Concevoir et évaluer quantitativement** (via le **F1-score de détection de contours**) un pipeline complet de débruitage, rehaussement et extraction de contours en présence de bruit impulsionnel (poivre et sel) et gaussien.

## Étape 0 : Préparation de l'environnement et chargement des données
Importer les bibliothèques nécessaires (`time`, `numpy`, `opencv-python`, `matplotlib`, `scikit-image`) et vérifier l'image de référence.

In [ ]:
import time
import numpy as np
import cv2
import matplotlib.pyplot as plt
from skimage import data

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['image.cmap'] = 'gray'

img_ref = data.camera()
print(f"Image chargée : {img_ref.shape} | Type : {img_ref.dtype}")

---
## Étape 1 : Convolution 2D, filtre moyenneur et noyaux Gaussiens séparables

### 1.1 Corrélation croisée vs Convolution 2D (*From Scratch*)
**Rappel théorique :** Pour un noyau $h$ de dimension $(2a+1) \times (2b+1)$ :
- **Corrélation croisée :** $(h \otimes f)(r, c) = \sum_{u=-a}^{a} \sum_{v=-b}^{b} h(u, v)\, f(r + u, c + v)$
- **Convolution 2D :** $(h * f)(r, c) = \sum_{u=-a}^{a} \sum_{v=-b}^{b} h(u, v)\, f(r - u, c - v)$ (équivalent à une corrélation croisée avec le noyau retourné de $180^\circ$ : `np.flip(h)`).

**Consignes :**
1. Compléter la fonction `convolution_2d(img, kernel, pad_mode='reflect')` en calculant en `float64`.
2. Valider votre fonction sur la matrice $10 \times 10$ de référence (carré d'intensité 90) avec un filtre moyenneur $3 \times 3$ (`np.ones((3, 3)) / 9.0`).

In [ ]:
# ======================================================================
# ESPACE D'IMPLÉMENTATION ÉTUDIANT — ÉTAPE 1.1
# ======================================================================

def convolution_2d(img, kernel, pad_mode='reflect'):
    """
    Applique une convolution 2D discrète entre img (H, W) et kernel (kh, kw).
    Retourne une matrice float64 de même dimension (H, W).
    """
    kh, kw = kernel.shape
    pad_h, pad_w = kh // 2, kw // 2
    # VOTRE CODE ICI : retourner le noyau de 180°, padder l'image avec np.pad, puis convoluer
    out = None
    return out

# --- Validation sur la matrice 10x10 de référence ---
f_test = np.zeros((10, 10), dtype=np.float64)
f_test[2:8, 2:8] = 90.0
f_test[2, 5:8] = 0.0; f_test[3, 6:8] = 0.0; f_test[4, 7] = 0.0; f_test[7, 2] = 0.0; f_test[7, 7] = 0.0
box_3x3 = np.ones((3, 3), dtype=np.float64) / 9.0

# VOTRE CODE ICI : Appliquer convolution_2d(f_test, box_3x3, pad_mode='constant') et afficher la matrice résultante

### 1.2 Discrétisation d'un noyau Gaussien 2D et démonstration de la séparabilité
**Rappel théorique :** Le filtre Gaussien isotrope 2D s'écrit :
$$G(x, y) = \frac{1}{2\pi\sigma^2} \exp\left(-\frac{x^2 + y^2}{2\sigma^2}\right) = \underbrace{\left(\frac{1}{\sqrt{2\pi}\sigma}e^{-\frac{x^2}{2\sigma^2}}\right)}_{G_1(x)} \cdot \underbrace{\left(\frac{1}{\sqrt{2\pi}\sigma}e^{-\frac{y^2}{2\sigma^2}}\right)}_{G_1(y)}$$
Cette séparabilité réduit la complexité par pixel de $\mathcal{O}(K^2)$ à $\mathcal{O}(2K)$.

**Consignes :**
1. Implémenter `generer_noyau_gaussien(taille, sigma)` qui retourne `(K_2d, k_1d)` normalisés de somme égale à 1.
2. Vérifier que pour `taille=3, sigma=1.0`, la normalisation entière par le coin supérieur gauche (`np.round(K_2d / K_2d[0, 0])`) donne la matrice entière classique $\begin{bmatrix} 1 & 2 & 1 \\ 2 & 4 & 2 \\ 1 & 2 & 1 \end{bmatrix}$.
3. Implémenter `convolution_separable(img, k_1d)` (passe horizontale $1 \times K$ suivie d'une passe verticale $K \times 1$) et comparer la **précision numérique** (erreur maximale absolue) et le **facteur d'accélération (*speedup*)** face à `convolution_2d` pour $K \in \{3, 7, 11, 15\}$.

In [ ]:
# ======================================================================
# ESPACE D'IMPLÉMENTATION ÉTUDIANT — ÉTAPE 1.2
# ======================================================================

def generer_noyau_gaussien(taille, sigma):
    """
    Génère le noyau Gaussien 2D (taille x taille) et le vecteur 1D associé (taille,),
    tous deux normalisés pour que leur somme vaille 1.
    """
    assert taille % 2 == 1, "La taille du noyau doit être impaire."
    # VOTRE CODE ICI
    K_2d = None
    k_1d = None
    return K_2d, k_1d

def convolution_separable(img, k_1d, pad_mode='reflect'):
    """
    Applique deux convolutions 1D successives (horizontale puis verticale).
    """
    # VOTRE CODE ICI
    out = None
    return out

# VOTRE CODE ICI :
# 1. Afficher les masques entiers pour taille=3 et taille=5 (sigma=1.0)
# 2. Mesurer les temps d'exécution et tracer le speedup (T_2D / T_sep) pour K = [3, 7, 11, 15]

---
## Étape 2 : Rehaussement de détails et détection de contours

### 2.1 Filtrage Laplacien et *Unsharp Masking*
**Rappel théorique :**
- **Laplacien discret :** $\nabla^2 f = \frac{\partial^2 f}{\partial x^2} + \frac{\partial^2 f}{\partial y^2}$, approché par les masques à somme nulle :
  $$L_{-4} = \begin{bmatrix} 0 & 1 & 0 \\ 1 & -4 & 1 \\ 0 & 1 & 0 \end{bmatrix}, \qquad L_{-8} = \begin{bmatrix} 1 & 1 & 1 \\ 1 & -8 & 1 \\ 1 & 1 & 1 \end{bmatrix}$$
  Le rehaussement s'obtient par soustraction (car le coefficient central est négatif) : $I_{\text{sharp}} = \text{clip}(I - c \cdot (I * L), 0, 255)$.
- ***Unsharp Masking* (masque flou) :** $I_{\text{sharp}} = \text{clip}\left(I + k \cdot (I - I * G_{\sigma}),\; 0,\; 255\right)$ avec $k > 0$ ($k=1$ : *Unsharp Masking*, $k>1$ : *High-Boost Filtering*).

**Consignes :**
1. Implémenter `unsharp_masking(img, taille=5, sigma=1.5, k=1.0)`.
2. Comparer visuellement sur `img_ref` le rehaussement par Laplacien $L_{-8}$ ($c=0.5$) et par *Unsharp Masking* ($k=1.5$), en affichant également la carte des hautes fréquences extraites.

In [ ]:
# ======================================================================
# ESPACE D'IMPLÉMENTATION ÉTUDIANT — ÉTAPE 2.1
# ======================================================================

def unsharp_masking(img, taille=5, sigma=1.5, k=1.0):
    """
    Rehausse les contours par soustraction d'une version lissée par filtre Gaussien.
    Retourne : (img_sharp uint8, masque_hf float64)
    """
    # VOTRE CODE ICI
    img_sharp = None
    masque_hf = None
    return img_sharp, masque_hf

# VOTRE CODE ICI : Comparer Laplacien L_-8 et Unsharp Masking sur data.camera()

### 2.2 Opérateurs de Sobel : Magnitude du gradient et carte d'orientation
**Rappel théorique :** Les noyaux de dérivation lissée de Sobel s'écrivent :
$$S_x = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix} \text{ (contours verticaux)}, \qquad S_y = \begin{bmatrix} -1 & -2 & -1 \\ 0 & 0 & 0 \\ 1 & 2 & 1 \end{bmatrix} \text{ (contours horizontaux)}$$
avec la magnitude $M(r,c) = \sqrt{G_x^2 + G_y^2}$ et l'orientation $\theta(r,c) = \text{atan2}(G_y, G_x) \in [-\pi, \pi]$.

**Consignes :**
1. Implémenter `sobel_gradients(img)` retournant `(Gx, Gy, magnitude, orientation)`.
2. Générer une visualisation couleur HSV de l'orientation des contours où la **teinte (Hue)** encode $\theta$ et la **valeur (Value)** encode la magnitude $M$ normalisée.

In [ ]:
# ======================================================================
# ESPACE D'IMPLÉMENTATION ÉTUDIANT — ÉTAPE 2.2
# ======================================================================

def sobel_gradients(img):
    """
    Calcule les gradients horizontaux Gx, verticaux Gy, la magnitude M et l'orientation theta.
    """
    Sx = np.array([[-1, 0, 1],
                   [-2, 0, 2],
                   [-1, 0, 1]], dtype=np.float64)
    Sy = np.array([[-1, -2, -1],
                   [ 0,  0,  0],
                   [ 1,  2,  1]], dtype=np.float64)
    # VOTRE CODE ICI
    Gx, Gy, mag, theta = None, None, None, None
    return Gx, Gy, mag, theta

# VOTRE CODE ICI : Afficher Gx, Gy, la magnitude M et la carte couleur HSV d'orientation

---
## Étape 3 : Étude de cas expérimentale — Pipeline complet de détection de contours en milieu bruité

En vision industrielle, les capteurs CCD/CMOS sont sujets au **bruit impulsionnel (poivre et sel)** et au **bruit thermique gaussien**, qui génèrent de nombreux faux contours lorsqu'un opérateur dérivateur (Sobel/Laplacien) est appliqué directement.

**Protocole expérimental :**
1. Générer la carte de contours de référence `edges_gt` à partir de l'image propre `data.coins()` (Sobel + seuillage d'Otsu sur la magnitude).
2. Dégrader `data.coins()` avec un **bruit impulsionnel (poivre et sel à 6%) + bruit gaussien ($\sigma=8$)**.
3. Implémenter et comparer **3 pipelines d'extraction de contours** :
   - **P1 (Naïf) :** Sobel direct sur l'image bruitée + seuillage d'Otsu.
   - **P2 (Linéaire) :** Lissage Gaussien ($5\times 5, \sigma=1.2$) $\rightarrow$ Sobel $\rightarrow$ seuillage d'Otsu.
   - **P3 (Non linéaire + Rehaussement) :** **Filtre médian $3\times 3$** (`cv2.medianBlur`) $\rightarrow$ **Rehaussement Laplacien** ($I_{\text{sharp}} = I_{\text{med}} - 0.3 \cdot (I_{\text{med}} * L_{-4})$) $\rightarrow$ **Sobel** $\rightarrow$ **seuillage d'Otsu**.
4. Évaluer chaque pipeline par rapport à `edges_gt` en calculant la **Précision, le Rappel et le F1-score** des contours détectés.

In [ ]:
# ======================================================================
# ESPACE D'IMPLÉMENTATION ÉTUDIANT — ÉTAPE 3
# ======================================================================

def degrader_image_capteur(img, prob_sp=0.06, sigma_gauss=8.0, seed=42):
    rng = np.random.default_rng(seed)
    noisy = img.astype(np.float64) + rng.normal(0, sigma_gauss, img.shape)
    noisy = np.clip(noisy, 0, 255).astype(np.uint8)
    rnd = rng.random(img.shape)
    noisy[rnd < prob_sp / 2.0] = 0
    noisy[rnd > 1.0 - prob_sp / 2.0] = 255
    return noisy

def calculer_f1_contours(edges_pred, edges_gt):
    # VOTRE CODE ICI : Calculer Précision, Rappel et F1-score entre masques binaires
    pass

# VOTRE CODE ICI : Comparer P1, P2 et P3 sur data.coins() bruitée et afficher le tableau des F1-scores

---
## Question de synthèse

> **Problématique scientifique :**  
> Dans le cadre de la conception d'un système d'inspection métrologique sub-pixellique (mesure dimensionnelle de pièces mécaniques usinées ou détection de micro-fissures) :
> 1. **Analyse fréquentielle et non-linéarité :** Expliquez pourquoi un filtre linéaire passe-bas (moyenneur ou Gaussien) est structurellement inefficace face au bruit impulsionnel (poivre et sel) tout en dégradant la précision métrologique de localisation des bords, et justifiez mathématiquement pourquoi le **filtre médian** préserve les discontinuités franches (*step edges*) sans les déplacer.
> 2. **Du gradient de Sobel au détecteur optimal de Canny :** Quelles sont les deux limitations majeures d'un simple seuillage de la magnitude du gradient de Sobel ($M(r,c) > T$) pour extraire des contours exploitables géométriquement ? Expliquez comment les deux étapes supplémentaires introduites par **Canny** — la **suppression des non-maxima locaux (NMS)** le long de la direction $\theta(r,c)$ et le **seuillage par hystérésis** $(T_{\text{bas}}, T_{\text{haut}})$ — résolvent ces problèmes.
> 3. **Convergence vers les réseaux de neurones convolutifs (CNN) :** En quoi les couches convolutives d'un CNN (et les blocs de convolution séparable en profondeur / *Depthwise Separable Convolutions* utilisés dans MobileNet ou EfficientNet) généralisent-elles directement les principes de filtrage spatial et de séparabilité étudiés dans ce TP ?

### Rédigez votre synthèse argumentée dans la cellule ci-dessous :

**Réponse de l'étudiant :**

1. *Filtre linéaire vs filtre médian face au bruit impulsionnel et à la métrologie des bords :*  
   *(À compléter...)*

2. *Limites de Sobel seuillé et apport du détecteur de Canny (NMS + Hystérésis) :*  
   *(À compléter...)*

3. *Lien théorique entre filtrage spatial séparable et architectures CNN modernes :*  
   *(À compléter...)*